# 01 - Quickstart Synthetic

This notebook introduces the task-first data contract used by `alemtl`:

- a dataset item is balanced across tasks;
- a dataloader batch is shaped `(n_tasks, batch, n_features)`;
- targets are shaped `(n_tasks, batch, n_outputs)`.

The synthetic dataset contains three sine-like tasks with related but distinct response functions.

In [1]:
from pathlib import Path
import sys

repo_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "pyproject.toml").exists() and (path / "src" / "alemtl").exists()
    ),
    Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve(),
)
for path in (repo_root, repo_root / "src"):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

import math
import pandas as pd
import torch

from alemtl.data import MultitaskDataset, MultitaskDataloader

torch.manual_seed(7)

## Build a Synthetic Multi-task Frame

Each row belongs to one task. The `MultitaskDataset` groups rows by task and returns balanced samples.

In [2]:
def make_frame(n_per_task: int = 96) -> pd.DataFrame:
    x = torch.linspace(0, 2 * math.pi, steps=n_per_task)
    specs = {
        "low_freq": (1.0, 0.0),
        "phase_shift": (1.0, 0.6),
        "high_freq": (2.0, 0.2),
    }
    rows = []
    for task, (frequency, phase) in specs.items():
        target = torch.sin(frequency * x + phase)
        for xi, yi in zip(x, target):
            rows.append({
                "task": task,
                "x": float(xi),
                "sin_x": float(torch.sin(xi)),
                "cos_x": float(torch.cos(xi)),
                "target": float(yi),
            })
    return pd.DataFrame(rows)

data = make_frame()
data.head()

,task,x,sin_x,cos_x,target
0,low_freq,0.000000,0.000000,1.000000,0.000000
1,low_freq,0.066139,0.066091,0.997814,0.066091
2,low_freq,0.132278,0.131892,0.991264,0.131892
3,low_freq,0.198416,0.197117,0.980380,0.197117
4,low_freq,0.264555,0.261480,0.965209,0.261480


## Create Dataset and Dataloader

`MultitaskDataloader` permutes PyTorch's default `(batch, tasks, ...)` output into `(tasks, batch, ...)`, which is what the models expect.

In [3]:
dataset = MultitaskDataset(data, task_id="task", target_names="target")
loader = MultitaskDataloader(dataset, batch_size=16, shuffle=True, num_workers=0)

X, y = next(iter(loader))
print(dataset)
print("Task counts:", dataset.get_task_counts_by_original())
print("X shape:", tuple(X.shape))
print("y shape:", tuple(y.shape))

MultitaskDataset(n_tasks=3, n_features=3, n_targets=1, len=96)
Task counts: {'high_freq': 96, 'low_freq': 96, 'phase_shift': 96}
X shape: (3, 16, 3)
y shape: (3, 16, 1)


## Fit a Tiny Model for a Few Steps

This is not intended as a benchmark. It simply verifies that the data contract connects cleanly to `MultiTaskModel`.

In [6]:
from torch import nn
from alemtl.models import MultiTaskModel

model = MultiTaskModel(
    n_tasks=dataset.n_tasks,
    modules_layout={
        "trunk": {"shared": "hard", "module": lambda: nn.Sequential(nn.Linear(3, 16), nn.ReLU())},
        "head": {"shared": "soft", "module": lambda: nn.Linear(16, 1)},
    },
    similarity_layers={"in": "trunk", "out": "head"},
    same_parameters=True,
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

for step, (X_batch, y_batch) in zip(range(20), loader):
    optimizer.zero_grad()
    pred = model(X_batch)
    loss = loss_fn(pred, y_batch)
    loss.backward()
    optimizer.step()

print("last loss:", float(loss))

last loss: 0.36525753140449524


## Inspect Predictions

The first axis remains the task axis throughout the workflow.

In [5]:
with torch.no_grad():
    pred = model(X)

print("prediction shape:", tuple(pred.shape))
print("first task predictions:")
print(pred[0, :5, 0])

prediction shape: (3, 16, 1)
first task predictions:
tensor([-0.1672,  0.0797, -0.0071,  0.1189,  0.0724])
